# Hugging Face Usage Examples

This notebook shows how to load the released NSF-SciFy datasets and LoRA adapters directly from Hugging Face.

## Install Dependencies

The dataset cells only require `datasets`. Model loading uses `unsloth`, which should be installed following the environment used for training/evaluation in this repository.

In [ ]:
# Optional if datasets is not already installed:
# %pip install datasets

## Released Dataset Repositories

- `darpa-scify/nsf-scify-matsci`: materials-science/DMR NSF awards.
- `darpa-scify/nsf-scify-20k`: 20K all-awards NSF-SciFy subset.

Both repositories have `train`, `validation`, and `test` splits.

In [ ]:
from datasets import load_dataset

matsci = load_dataset("darpa-scify/nsf-scify-matsci")
twenty_k = load_dataset("darpa-scify/nsf-scify-20k")

print(matsci)
print(twenty_k)

In [ ]:
print("matsci columns:", matsci["train"].column_names)
print("20k columns:", twenty_k["train"].column_names)

example = twenty_k["train"][0]
for key in ["award_id", "title", "technical_abstract", "non_technical_abstract", "verifiable_claims", "investigation_proposals"]:
    print(f"\n{key}:\n{example.get(key)}")

## Combined Materials-Science Plus 20K Data

There is not a separate combined dataset repo. Load the two datasets separately and concatenate matching splits.

In [ ]:
from datasets import concatenate_datasets

combined = {
    split: concatenate_datasets([matsci[split], twenty_k[split]])
    for split in ["train", "validation", "test"]
}

for split, ds in combined.items():
    print(split, len(ds))

## Released Model Repositories

These are LoRA adapters trained from `unsloth/mistral-7b-instruct-v0.3`.

Materials-science/DMR adapters:

- `darpa-scify/nsf-scify-matsci-nta`
- `darpa-scify/nsf-scify-matsci-claims`
- `darpa-scify/nsf-scify-matsci-ip`

Materials-science plus 20K adapters:

- `darpa-scify/nsf-scify-matsci-20k-nta`
- `darpa-scify/nsf-scify-matsci-20k-claims`
- `darpa-scify/nsf-scify-matsci-20k-ip`

In [ ]:
MODEL_REPOS = {
    "matsci_nta": "darpa-scify/nsf-scify-matsci-nta",
    "matsci_claims": "darpa-scify/nsf-scify-matsci-claims",
    "matsci_ip": "darpa-scify/nsf-scify-matsci-ip",
    "matsci_20k_nta": "darpa-scify/nsf-scify-matsci-20k-nta",
    "matsci_20k_claims": "darpa-scify/nsf-scify-matsci-20k-claims",
    "matsci_20k_ip": "darpa-scify/nsf-scify-matsci-20k-ip",
}

MODEL_REPOS

## Load A Released Adapter

This loads the adapter with Unsloth, matching the repository training/evaluation code. Pick the adapter for the task you want to run.

In [ ]:
# This cell requires an Unsloth-capable GPU environment.
# from unsloth import FastLanguageModel

# model_name = MODEL_REPOS["matsci_20k_claims"]
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_name,
#     max_seq_length=2048,
#     dtype=None,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)

## Use Released Adapters With The Repository Scripts

The scripts accept Hugging Face adapter IDs through `--model`. Use matching datasets, tasks, and prompt modes. These shell examples assume the notebook kernel is running from the repository root.

In [ ]:
# Non-technical abstract generation
# !python scripts/gen.py \
#   --root_dir . \
#   --model darpa-scify/nsf-scify-matsci-20k-nta \
#   --dataset matsci_and_20k \
#   --task tech2nontech \
#   --prompt_mode tech2nontech_instruct_user_assistant \
#   --batch_size 1

In [ ]:
# Verifiable claim generation
# !python scripts/gen.py \
#   --root_dir . \
#   --model darpa-scify/nsf-scify-matsci-20k-claims \
#   --dataset matsci_and_20k \
#   --task technontech2claims \
#   --prompt_mode text2claims_instruct_user_assistant \
#   --batch_size 1

In [ ]:
# Investigation-proposal generation
# !python scripts/gen.py \
#   --root_dir . \
#   --model darpa-scify/nsf-scify-matsci-20k-ip \
#   --dataset matsci_and_20k \
#   --task technontech2ip \
#   --prompt_mode text2ip_instruct_user_assistant \
#   --batch_size 1

## Evaluate A Released Adapter

`scripts/eval.py` uses the same model IDs. Non-technical abstract evaluation uses BERTScore, ROUGE, and BLEU. Claims and investigation-proposal evaluation uses the LLM-judge metrics in `src/claims_utils.py`.

In [ ]:
# Non-technical abstract evaluation
# !python scripts/eval.py \
#   --root_dir . \
#   --model darpa-scify/nsf-scify-matsci-20k-nta \
#   --dataset matsci_and_20k \
#   --task tech2nontech \
#   --prompt_mode tech2nontech_instruct_user_assistant \
#   --batch_size 1

In [ ]:
# Claims evaluation. Set up the OpenAI API key before running LLM-judge metrics.
# !python scripts/eval.py \
#   --root_dir . \
#   --model darpa-scify/nsf-scify-matsci-20k-claims \
#   --dataset matsci_and_20k \
#   --task technontech2claims \
#   --prompt_mode text2claims_instruct_user_assistant \
#   --batch_size 1

In [ ]:
# Investigation-proposal evaluation. Set up the OpenAI API key before running LLM-judge metrics.
# !python scripts/eval.py \
#   --root_dir . \
#   --model darpa-scify/nsf-scify-matsci-20k-ip \
#   --dataset matsci_and_20k \
#   --task technontech2ip \
#   --prompt_mode text2ip_instruct_user_assistant \
#   --batch_size 1